# Exercise 1A: Label the Pipeline
Exercise: For the sentence "The decoder generated a summary", 
list a plausible sequence of stages from raw text to model prediction. 
Leave room for token ids, embeddings, attention, and output logits.

Solution: The Transformer Pipeline for Your Sentence
Raw Text Input
    ↓
1. TOKENIZATION
   "The decoder generated a summary" 
   → token_ids = [101, 1996, 24992, 2686, 1037, 6625, 102]
   (These are example token IDs from a BERT-like tokenizer)
   
   ↓
2. EMBEDDING LOOKUP
   token_ids → embedding vectors (shape: [7, 768])
   (Each token becomes a 768-dim dense vector)
   
   ↓
3. POSITION ENCODING
   Add positional information (so attention knows token order)
   embedding + position_embedding → positional embeddings
   
   ↓
4. SELF-ATTENTION
   Tokens compare with each other:
   - "The" attends to "decoder", "generated", "a", "summary"
   - "decoder" attends to "The", "generated", etc.
   Result: contextual vectors with enriched information
   
   ↓
5. FEED-FORWARD LAYERS
   Pass each contextual vector through dense transformations
   → hidden states (updated representations)
   
   ↓
6. PREDICTION HEAD
   hidden states → output logits
   (For classification: class logits; for generation: token logits)
   
   ↓
Output

# Exercise 2A: Why Mask Padding Tokens?
The Problem
When you have a batch with different sentence lengths:

* Sentence 1: "The model works" (3 words)
* Sentence 2: "I like transformers very much" (5 words)

Computers need everything to be the same size (for efficiency). So we pad the short sentence with fake tokens:

* Sentence 1: "The model works [PAD] [PAD]" (now 5 words)
* Sentence 2: "I like transformers very much" (5 words)

The Issue
During attention, these [PAD] tokens have no real meaning. If we don't mask them:

* "The" would attend to: "model" (real), "works" (real), [PAD] (FAKE!)
* The model wastes attention on meaningless padding

The Solution: Masking
We tell the attention mechanism: "Ignore [PAD] tokens completely. Set their attention weights to 0."



Padding tokens should be masked because:
1. They are artificial fillers with no semantic meaning
2. If not masked, they waste the model's attention capacity
3. The model might learn to attend to padding tokens,
   which could hurt its ability to understand real words
4. Masking ensures attention focuses only on actual content

# Exercise 2B: Interpret Attention
The Setup
The word "it" attends strongly to "model" means:

When the attention mechanism is computing what "it" should focus on, it gives a HIGH weight to "model".

What This Tells Us
Possible relationships:

1. Pronoun reference (most likely)
* "it" = referring back to "model"
* Example: "The model works well. It is efficient."
* The attention head learned: "it" often refers to nouns

2. Causal relationship
* "it" describes what happens because of the model
* Example: "The model generates text. It is useful for..."

3. Subject-object relationship
* "it" is the direct object of an action on the model
* Example: "We trained the model. It performed well."

Why This Matters
Attention heads learn different linguistic patterns:

* Some heads learn syntax (grammar)
* Some heads learn semantics (meaning)
* Some heads learn discourse (references between words)

The fact that "it" → "model" is a STRONG connection suggests this head learned pronoun resolution — how pronouns refer back to nouns.

The attention head is likely capturing a pronoun reference relationship.
"It" is probably a pronoun referring back to "model" 
(e.g., "The model works. It is efficient").
This shows the attention head learned that pronouns often depend on
their antecedent nouns, which is an important linguistic pattern.

# Mini lab: encoder classifier
1. Load rows where task_type equals encoder_classification.
2. Use input_text as the model input and label as the target.
3. Split by the split column: train, validation, test.
4. Train a classifier and evaluate accuracy on the test split.
5. Run inference on: "The model produced useful summaries."

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load and filter
df    = pd.read_csv('/Users/jhonlloydval/DataScience_BOOTCAMP/W3D2/transformers/transformer_workbook_dataset.csv')
train = df[(df['split'] == 'train')      & (df['task_type'] == 'encoder_classification')]
valid = df[(df['split'] == 'validation') & (df['task_type'] == 'encoder_classification')]
test  = df[(df['split'] == 'test')       & (df['task_type'] == 'encoder_classification')]

print(f"Train: {len(train)} | Valid: {len(valid)} | Test: {len(test)}")

label_map = {'positive': 0, 'negative': 1, 'neutral': 2}
id2label  = {v: k for k, v in label_map.items()}

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model     = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

# Training
model.train()
for epoch in range(3):
    total_loss = 0
    for _, row in train.iterrows():
        inputs = tokenizer(str(row['input_text']), return_tensors='pt',
                           truncation=True, max_length=128).to(device)
        label  = torch.tensor([label_map[str(row['label'])]]).to(device)
        out    = model(**inputs, labels=label)

        optimizer.zero_grad()
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()

    # Validation accuracy
    model.eval()
    correct = 0
    for _, row in valid.iterrows():
        inputs = tokenizer(str(row['input_text']), return_tensors='pt',
                           truncation=True, max_length=128).to(device)
        with torch.no_grad():
            pred = torch.argmax(model(**inputs).logits, dim=-1).item()
        if pred == label_map.get(str(row['label']), -1):
            correct += 1
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train):.4f} | Val Acc: {correct/len(valid):.0%}")
    model.train()

# Test accuracy
model.eval()
correct = 0
for _, row in test.iterrows():
    inputs = tokenizer(str(row['input_text']), return_tensors='pt',
                       truncation=True, max_length=128).to(device)
    with torch.no_grad():
        pred = torch.argmax(model(**inputs).logits, dim=-1).item()
    if pred == label_map.get(str(row['label']), -1):
        correct += 1

print(f"\nTest Accuracy: {correct}/{len(test)} = {correct/len(test):.0%}")

# Three example predictions
print("\nThree Example Predictions:")
print("-" * 60)
for _, row in test.head(3).iterrows():
    inputs = tokenizer(str(row['input_text']), return_tensors='pt',
                       truncation=True, max_length=128).to(device)
    with torch.no_grad():
        pred = torch.argmax(model(**inputs).logits, dim=-1).item()
    print(f"Text:      {row['input_text']}")
    print(f"True:      {row['label']}")
    print(f"Predicted: {id2label[pred]}\n")


Train: 8 | Valid: 2 | Test: 2


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6204.96it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1 | Loss: 1.1152 | Val Acc: 50%
Epoch 2 | Loss: 1.0197 | Val Acc: 50%
Epoch 3 | Loss: 0.9474 | Val Acc: 50%

Test Accuracy: 2/2 = 100%

Three Example Predictions:
------------------------------------------------------------
Text:      The generated answer ignored the instruction.
True:      negative
Predicted: negative

Text:      The dataset contains inputs and targets.
True:      neutral
Predicted: neutral



: 

# REFLECTION

Bidirectional encoders can see context from both directions simultaneously, allowing them to understand full meaning rather than predicting sequentially. They produce a fixed representation of the entire text that's perfect for classification tasks—just feed the output into a classifier head. Their pre-training on large corpora gives them strong linguistic knowledge, and since classification needs a single decision per document (not token-by-token), bidirectional models are more efficient and effective than autoregressive alternatives.

# Exercise 4A

In one paragraph, explain why a decoder should not see future target tokens during training.


A decoder should not see future target tokens during training because it would create a mismatch between training and inference that would break the model's ability to work in the real world. During training, the decoder's job is to predict the next token given only the previous tokens (this is called next-token prediction). If the model could peek at future tokens during training, it would learn to memorize or directly copy them rather than actually learning to predict what comes next based on context. However, during inference, the model only has access to tokens it has already generated, so it cannot cheat. This training-inference mismatch would cause the model to perform poorly when deployed—it would have learned a strategy that only works when you can see the answer. Causal masking (hiding future tokens) during training enforces the same constraint that exists during inference, ensuring the model learns genuine predictive patterns that will work when generating new text token-by-token.

# Mini Lab: Decoder Generation

1. Load rows where `task_type` equals `decoder_generation`.
2. Format each row as `prompt + target_text`.
3. Train or fine-tune a small causal language model on the training rows.
4. At inference, provide only the prompt and ask the model to continue.
5. Compare greedy decoding with sampling.

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Step 1: Load rows where task_type equals decoder_generation
df         = pd.read_csv('/Users/jhonlloydval/DataScience_BOOTCAMP/W3D2/transformers/transformer_workbook_dataset.csv')
decoder_df = df[(df['task_type'] == 'decoder_generation') & (df['split'] == 'train')]
test_df    = df[(df['task_type'] == 'decoder_generation') & (df['split'] == 'test')]
print(f"Train rows: {len(decoder_df)} | Test rows: {len(test_df)}")

# Step 2: Format each row as prompt + target_text
training_texts = [f"{row['input_text']} {row['target_text']}" for _, row in decoder_df.iterrows()]
print(f"Example: {training_texts[0][:120]}...")

# Step 3: Load model and fine-tune
tokenizer = AutoTokenizer.from_pretrained('distilgpt2')
model     = AutoModelForCausalLM.from_pretrained('distilgpt2')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

model.train()
for epoch in range(2):
    total_loss = 0
    for text in training_texts[:20]:
        inputs   = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).to(device)
        outputs  = model(input_ids=inputs['input_ids'], labels=inputs['input_ids'])
        optimizer.zero_grad()
        outputs.loss.backward()
        optimizer.step()
        total_loss += outputs.loss.item()
    print(f"Epoch {epoch+1} | Loss: {total_loss/20:.4f}")

model.eval()

# Step 4 & 5: Inference — prompt only; compare greedy vs sampling
print("\nGreedy vs Sampling Comparison:")
print("-" * 60)
for _, row in test_df.head(3).iterrows():
    prompt    = str(row['input_text'])
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        greedy = model.generate(input_ids, max_length=60, do_sample=False)
        sample = model.generate(input_ids, max_length=60, do_sample=True, temperature=0.7)

    print(f"Prompt:   {prompt}")
    print(f"Greedy:   {tokenizer.decode(greedy[0], skip_special_tokens=True)}")
    print(f"Sampling: {tokenizer.decode(sample[0], skip_special_tokens=True)}\n")


Train rows: 6 | Test rows: 1
Example: Write one sentence about self-attention. Self-attention lets each token gather information from other tokens in the same...


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 6661.38it/s]
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1 | Loss: 1.4366


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


# Exercise 5A

Mark each task as **encoder-only**, **decoder-only**, or **encoder-decoder**:

| Task | Architecture |
|---|---|
| Spam classification | Encoder-only — classifies a fixed input, no generation needed |
| Article summarization | Encoder-Decoder — reads full article, generates shorter output |
| Story continuation | Decoder-only — generates new tokens from a prompt |
| Translation | Encoder-Decoder — reads source language, generates target language |
| Retrieval embedding | Encoder-only — produces a fixed vector representing the input |

# Mini Lab: Sequence-to-Sequence (Encoder-Decoder)

1. Load rows where `task_type` starts with `encoder_decoder`.
2. Use `input_text` as the encoder source and `target_text` as the decoder target.
3. Train with teacher forcing.
4. At inference, encode the source once and decode target tokens step by step.
5. Evaluate exact match for pseudo-code rows and qualitative quality for summaries.

In [ ]:
import pandas as pd
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Step 1: Load rows where task_type starts with encoder_decoder
df               = pd.read_csv('/Users/jhonlloydval/DataScience_BOOTCAMP/W3D2/transformers/transformer_workbook_dataset.csv')
encoder_decoder_df = df[df['task_type'].str.startswith('encoder_decoder')]
train_df         = encoder_decoder_df[encoder_decoder_df['split'] == 'train'].head(20)
test_df          = encoder_decoder_df[encoder_decoder_df['split'] == 'test'].head(5)
print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")

# Step 2: input_text = encoder source, target_text = decoder target
print(f"Example input:  {train_df.iloc[0]['input_text']}")
print(f"Example target: {train_df.iloc[0]['target_text']}\n")

tokenizer = T5Tokenizer.from_pretrained('t5-small')
model     = T5ForConditionalGeneration.from_pretrained('t5-small')
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

# Step 3: Train with teacher forcing (labels= handles it internally)
model.train()
for epoch in range(2):
    total_loss = 0
    for _, row in train_df.iterrows():
        source = tokenizer(row['input_text'], return_tensors='pt',
                           truncation=True, max_length=128).to(device)
        target = tokenizer(row['target_text'], return_tensors='pt',
                           truncation=True, max_length=64).to(device)
        out = model(input_ids=source['input_ids'],
                    attention_mask=source['attention_mask'],
                    labels=target['input_ids'])
        optimizer.zero_grad()
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_df):.4f}")

model.eval()

# Step 4 & 5: Inference + evaluation
print("\nSource → Target → Prediction:")
print("-" * 70)
for _, row in test_df.iterrows():
    source = tokenizer(row['input_text'], return_tensors='pt',
                       truncation=True, max_length=128).to(device)
    with torch.no_grad():
        output_ids = model.generate(source['input_ids'], max_length=64)
    prediction = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    print(f"Task:       {row['task_type']}")
    print(f"Input:      {row['input_text'][:100]}...")
    print(f"Target:     {row['target_text']}")
    print(f"Prediction: {prediction}")

    if 'pseudo' in row['task_type']:
        print(f"Exact Match: {row['target_text'].strip() == prediction.strip()}")
    else:
        overlap = len(set(row['target_text'].lower().split()) & set(prediction.lower().split())) \
                  / max(len(row['target_text'].split()), 1)
        print(f"Word Overlap: {overlap:.0%}")
    print()


# Baseline Training Checklist

- Start with a tiny subset and overfit it. This catches data formatting problems.
- Check that labels and targets are not empty for the selected task.
- Keep train, validation, and test examples separate.
- Track training loss and validation performance.
- Save the tokenizer, model weights, and label mapping together.

## Exercise 6A

Write a validation rule that checks whether every classification row has a non-empty `label` and every generation row has a non-empty `target_text`.

In [ ]:
import pandas as pd
import torch
import json, os
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load and split data
df    = pd.read_csv('/Users/jhonlloydval/DataScience_BOOTCAMP/W3D2/transformers/transformer_workbook_dataset.csv')
train = df[df['split'] == 'train']
valid = df[df['split'] == 'validation']
test  = df[df['split'] == 'test']
enc   = train[train['task_type'] == 'encoder_classification']

print(f"Train: {len(train)} | Valid: {len(valid)} | Test: {len(test)}")

# Exercise 6A: Validation rule
def validate_rows(df):
    classification_rows = df[df['task_type'] == 'encoder_classification']
    generation_rows     = df[df['task_type'].str.startswith('encoder_decoder') |
                             (df['task_type'] == 'decoder_generation')]
    bad_labels  = classification_rows[classification_rows['label'].isna() |
                                      (classification_rows['label'].astype(str).str.strip() == '')]
    bad_targets = generation_rows[generation_rows['target_text'].isna() |
                                  (generation_rows['target_text'].astype(str).str.strip() == '')]
    print(f"  Missing labels:      {len(bad_labels)}")
    print(f"  Missing target_text: {len(bad_targets)}")
    if len(bad_labels) == 0 and len(bad_targets) == 0:
        print("  All rows passed validation!\n")

print("Checklist 1: Validating data...")
validate_rows(train)

# Checklist 1: Overfit a tiny subset first
tiny = enc.head(5)
print(f"Checklist 2: Overfitting {len(tiny)} rows to test the pipeline...")

label_map = {'positive': 0, 'negative': 1, 'neutral': 2}
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model     = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

# Checklist 4: Track training loss and validation performance
model.train()
for epoch in range(3):
    total_loss = 0
    for _, row in tiny.iterrows():
        inputs = tokenizer(str(row['input_text']), return_tensors='pt',
                           truncation=True, max_length=128).to(device)
        label  = torch.tensor([label_map.get(str(row['label']), 0)]).to(device)
        out    = model(**inputs, labels=label)
        optimizer.zero_grad()
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    print(f"  Epoch {epoch+1} | Train Loss: {total_loss/len(tiny):.4f}")

# Checklist 5: Save tokenizer, model weights, and label mapping together
save_dir = '/Users/jhonlloydval/DataScience_BOOTCAMP/W3D2/saved_model'
os.makedirs(save_dir, exist_ok=True)
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
with open(f'{save_dir}/label_map.json', 'w') as f:
    json.dump(label_map, f)
print(f"\nChecklist 5: Model, tokenizer, and label map saved to: {save_dir}")


# Section 7: Inference Workflow

| Architecture | Inference pattern | Example output |
|---|---|---|
| Encoder-only | Single forward pass | `positive` |
| Decoder-only | Autoregressive loop | generated continuation |
| Encoder-decoder | Encode once, decode step by step | summary or translated target |

**Decoding options:**

| Method | Idea | Tradeoff |
|---|---|---|
| Greedy | Pick highest-probability token each step | Fast but can be dull or repetitive |
| Sampling | Randomly sample from a probability distribution | More varied but less predictable |
| Beam search | Keep multiple candidate sequences | Often stronger but slower |

## Exercise 7A

**Risk of using training target text during inference:**  
This is called *teacher forcing leakage*. The model never learns to recover from its own mistakes. At inference there is no ground truth — the model feeds its own (possibly wrong) predictions back in. Errors compound step by step, causing outputs to drift. The model looks accurate during training but performs much worse in reality.

In [ ]:
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    T5Tokenizer, T5ForConditionalGeneration
)

encoder_prompt = "The summary was concise and correct."
decoder_prompt = "Write one sentence about cross-attention."
enc_dec_prompt = "Transformers compare tokens with attention and use masks for generation."

# 1. Encoder-only: single forward pass → label
print("1. ENCODER-ONLY — Single forward pass")
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model     = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)
model.eval()
inputs = tokenizer(encoder_prompt, return_tensors='pt', truncation=True, max_length=128)
with torch.no_grad():
    logits = model(**inputs).logits
label_map  = {0: 'positive', 1: 'negative', 2: 'neutral'}
print(f"Input:  {encoder_prompt}")
print(f"Output: {label_map[torch.argmax(logits, dim=-1).item()]}\n")

# 2. Decoder-only: autoregressive loop → greedy, sampling, beam search
print("2. DECODER-ONLY — Autoregressive loop")
tokenizer = AutoTokenizer.from_pretrained('distilgpt2')
model     = AutoModelForCausalLM.from_pretrained('distilgpt2')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.eval()
input_ids = tokenizer.encode(decoder_prompt, return_tensors='pt')

with torch.no_grad():
    greedy = model.generate(input_ids, max_length=60, do_sample=False)
    sample = model.generate(input_ids, max_length=60, do_sample=True, temperature=0.7)
    beam   = model.generate(input_ids, max_length=60, num_beams=4, early_stopping=True)

print(f"Prompt:      {decoder_prompt}")
print(f"Greedy:      {tokenizer.decode(greedy[0], skip_special_tokens=True)}")
print(f"Sampling:    {tokenizer.decode(sample[0], skip_special_tokens=True)}")
print(f"Beam search: {tokenizer.decode(beam[0], skip_special_tokens=True)}\n")

# 3. Encoder-decoder: encode once, decode step by step
print("3. ENCODER-DECODER — Encode once, decode step by step")
tokenizer = T5Tokenizer.from_pretrained('t5-small')
model     = T5ForConditionalGeneration.from_pretrained('t5-small')
model.eval()
input_ids = tokenizer("summarize: " + enc_dec_prompt,
                      return_tensors='pt', truncation=True, max_length=128)['input_ids']
with torch.no_grad():
    output_ids = model.generate(input_ids, max_length=64)
print(f"Input:  {enc_dec_prompt}")
print(f"Output: {tokenizer.decode(output_ids[0], skip_special_tokens=True)}")


# Section 8: Mini Projects

| Project | Goal | Minimum deliverable |
|---|---|---|
| A. Encoder classifier | Predict sentiment-like labels | Test accuracy and three example predictions |
| B. Decoder prompt responder | Generate one-sentence explanations | Three prompts and generated completions |
| C. Seq2seq summarizer | Summarize transformer concept paragraphs | Three source-target-output comparisons |
| D. Seq2seq pseudo-code translator | Map instructions to pseudo-code | Exact-match score and error analysis |

---

## Project A: Encoder Classifier

**Experiment Log:**
- Model: `distilbert-base-uncased`
- Task rows: `encoder_classification`
- Epochs: 3 | LR: 2e-5 | Metric: Accuracy

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

df    = pd.read_csv('/Users/jhonlloydval/DataScience_BOOTCAMP/W3D2/transformers/transformer_workbook_dataset.csv')
train = df[(df['split'] == 'train')      & (df['task_type'] == 'encoder_classification')]
valid = df[(df['split'] == 'validation') & (df['task_type'] == 'encoder_classification')]
test  = df[(df['split'] == 'test')       & (df['task_type'] == 'encoder_classification')]

label_map = {'positive': 0, 'negative': 1, 'neutral': 2}
id2label  = {v: k for k, v in label_map.items()}

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model     = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

model.train()
for epoch in range(3):
    total_loss = 0
    for _, row in train.iterrows():
        inputs = tokenizer(str(row['input_text']), return_tensors='pt',
                           truncation=True, max_length=128).to(device)
        label  = torch.tensor([label_map[str(row['label'])]]).to(device)
        out    = model(**inputs, labels=label)
        optimizer.zero_grad()
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train):.4f}")

model.eval()
correct = 0
for _, row in test.iterrows():
    inputs = tokenizer(str(row['input_text']), return_tensors='pt',
                       truncation=True, max_length=128).to(device)
    with torch.no_grad():
        pred = torch.argmax(model(**inputs).logits, dim=-1).item()
    if pred == label_map.get(str(row['label']), -1):
        correct += 1
print(f"\nTest Accuracy: {correct}/{len(test)} = {correct/len(test):.0%}")

print("\nThree Example Predictions:")
for _, row in test.head(3).iterrows():
    inputs = tokenizer(str(row['input_text']), return_tensors='pt',
                       truncation=True, max_length=128).to(device)
    with torch.no_grad():
        pred = torch.argmax(model(**inputs).logits, dim=-1).item()
    print(f"Text:      {row['input_text']}")
    print(f"True:      {row['label']}  |  Predicted: {id2label[pred]}\n")


## Project B: Decoder Prompt Responder

**Experiment Log:**
- Model: `distilgpt2`
- Task rows: `decoder_generation`
- Epochs: 2 | LR: 5e-5 | Metric: Qualitative fluency

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

df       = pd.read_csv('/Users/jhonlloydval/DataScience_BOOTCAMP/W3D2/transformers/transformer_workbook_dataset.csv')
train    = df[(df['split'] == 'train') & (df['task_type'] == 'decoder_generation')]
test     = df[(df['split'] == 'test')  & (df['task_type'] == 'decoder_generation')]

tokenizer = AutoTokenizer.from_pretrained('distilgpt2')
model     = AutoModelForCausalLM.from_pretrained('distilgpt2')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

model.train()
for epoch in range(2):
    total_loss = 0
    for _, row in train.iterrows():
        text   = f"{row['input_text']} {row['target_text']}"
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).to(device)
        out    = model(input_ids=inputs['input_ids'], labels=inputs['input_ids'])
        optimizer.zero_grad()
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train):.4f}")

model.eval()
print("\nThree Prompts and Generated Completions:")
for _, row in test.head(3).iterrows():
    input_ids = tokenizer.encode(str(row['input_text']), return_tensors='pt').to(device)
    with torch.no_grad():
        output = model.generate(input_ids, max_length=60, do_sample=True, temperature=0.7)
    print(f"Prompt:    {row['input_text']}")
    print(f"Expected:  {row['target_text']}")
    print(f"Generated: {tokenizer.decode(output[0], skip_special_tokens=True)}\n")


## Project C: Seq2seq Summarizer

**Experiment Log:**
- Model: `t5-small`
- Task rows: `encoder_decoder_summarization`
- Epochs: 2 | LR: 3e-4 | Metric: Word overlap

In [ ]:
import pandas as pd
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

df    = pd.read_csv('/Users/jhonlloydval/DataScience_BOOTCAMP/W3D2/transformers/transformer_workbook_dataset.csv')
train = df[(df['split'] == 'train') & (df['task_type'] == 'encoder_decoder_summarization')]
test  = df[(df['split'] == 'test')  & (df['task_type'] == 'encoder_decoder_summarization')]

tokenizer = T5Tokenizer.from_pretrained('t5-small')
model     = T5ForConditionalGeneration.from_pretrained('t5-small')
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

model.train()
for epoch in range(2):
    total_loss = 0
    for _, row in train.iterrows():
        source = tokenizer("summarize: " + str(row['input_text']), return_tensors='pt',
                           truncation=True, max_length=128).to(device)
        target = tokenizer(str(row['target_text']), return_tensors='pt',
                           truncation=True, max_length=64).to(device)
        out = model(input_ids=source['input_ids'], attention_mask=source['attention_mask'],
                    labels=target['input_ids'])
        optimizer.zero_grad()
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train):.4f}")

model.eval()
print("\nThree Source-Target-Output Comparisons:")
for _, row in test.head(3).iterrows():
    source = tokenizer("summarize: " + str(row['input_text']), return_tensors='pt',
                       truncation=True, max_length=128).to(device)
    with torch.no_grad():
        output_ids = model.generate(source['input_ids'], max_length=64)
    output  = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    overlap = len(set(str(row['target_text']).lower().split()) & set(output.lower().split())) \
              / max(len(str(row['target_text']).split()), 1)
    print(f"Source:   {row['input_text'][:100]}...")
    print(f"Target:   {row['target_text']}")
    print(f"Output:   {output}")
    print(f"Overlap:  {overlap:.0%}\n")


## Project D: Seq2seq Pseudo-code Translator

**Experiment Log:**
- Model: `t5-small`
- Task rows: `encoder_decoder_translation`
- Epochs: 3 | LR: 3e-4 | Metric: Exact match + error analysis

In [ ]:
import pandas as pd
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

df    = pd.read_csv('/Users/jhonlloydval/DataScience_BOOTCAMP/W3D2/transformers/transformer_workbook_dataset.csv')
train = df[(df['split'] == 'train')      & (df['task_type'] == 'encoder_decoder_translation')]
test  = df[(df['split'] == 'test')       & (df['task_type'] == 'encoder_decoder_translation')]
if len(test) == 0:
    test = df[(df['split'] == 'validation') & (df['task_type'] == 'encoder_decoder_translation')]

print(f"Train: {len(train)} | Test: {len(test)}")

tokenizer = T5Tokenizer.from_pretrained('t5-small')
model     = T5ForConditionalGeneration.from_pretrained('t5-small')
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

model.train()
for epoch in range(3):
    total_loss = 0
    for _, row in train.iterrows():
        source = tokenizer(str(row['input_text']), return_tensors='pt',
                           truncation=True, max_length=64).to(device)
        target = tokenizer(str(row['target_text']), return_tensors='pt',
                           truncation=True, max_length=32).to(device)
        out = model(input_ids=source['input_ids'], attention_mask=source['attention_mask'],
                    labels=target['input_ids'])
        optimizer.zero_grad()
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train):.4f}")

model.eval()
exact_matches = 0
errors        = []

for _, row in test.iterrows():
    source = tokenizer(str(row['input_text']), return_tensors='pt',
                       truncation=True, max_length=64).to(device)
    with torch.no_grad():
        output_ids = model.generate(source['input_ids'], max_length=32)
    prediction = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    target     = str(row['target_text']).strip()
    match      = prediction.strip() == target
    if match:
        exact_matches += 1
    else:
        errors.append({'input': row['input_text'], 'target': target, 'prediction': prediction})
    print(f"Input:     {row['input_text']}")
    print(f"Target:    {target}")
    print(f"Predicted: {prediction}  |  Match: {match}\n")

print(f"Exact Match Score: {exact_matches}/{len(test)} = {exact_matches/max(len(test),1):.0%}")

if errors:
    print("\nError Analysis:")
    for err in errors:
        missed = [w for w in err['target'].split() if w not in err['prediction'].split()]
        extra  = [w for w in err['prediction'].split() if w not in err['target'].split()]
        print(f"  Input:   {err['input']}")
        print(f"  Missing: {missed}  |  Extra: {extra}\n")


# Experiment Log Template

| Field | Your notes |
|---|---|
| Model | |
| Task rows used | |
| Tokenizer settings | |
| Epochs | |
| Learning rate | |
| Metric | |
| Best result | |
| Most interesting error | |

# Section 9: Dataset Reference

The dataset is intentionally small and educational. It is not meant to produce a production-quality model. Its purpose is to help you test end-to-end training and inference pipelines without needing a large external corpus.

## CSV Schema

| Column | Description |
|---|---|
| id | Stable row identifier |
| task_type | One of `encoder_classification`, `decoder_generation`, `encoder_decoder_summarization`, `encoder_decoder_translation` |
| split | `train`, `validation`, or `test` |
| input_text | Source text, prompt, or classification input |
| target_text | Generation target for decoder and encoder-decoder rows |
| label | Classification label for encoder rows |

## Sample Rows (abbrev)

| id | task_type | split | input_text | target_text | label |
|---|---|---|---|---|---|
| 001 | encoder_classification | train | The model finished training faster than expected. |  | positive |
| 002 | encoder_classification | train | The validation loss kept increasing after epoch three. |  | negative |
| 003 | encoder_classification | train | The tokenizer loaded with the default settings. |  | neutral |
| 004 | encoder_classification | train | The inference results were accurate and consistent. |  | positive |
| 015 | encoder_decoder_summarization | train | A decoder generates tokens one at a time while masking future positions... | Decoders generate masked next tokens for text generation... |  |
| 016 | encoder_decoder_summarization | train | An encoder-decoder model first encodes the source text and then decodes... | Encoder-decoder models support sequence-to-sequence generation... |  |
| 017 | encoder_decoder_summarization | train | Teacher forcing trains a decoder by providing the previous gold target... | Teacher forcing uses gold previous tokens during decoding... |  |
| 018 | encoder_decoder_summarization | train | Positional information is needed because attention alone does not know... | Position signals help attention understand token order. |  |
| 035 | decoder_generation | train | Write one sentence about inference. | Inference uses a trained model to produce predictions... |  |
| 036 | decoder_generation | train | Write one sentence about cross-attention. | Cross-attention lets a decoder attend to encoder outputs... |  |
| 037 | decoder_generation | validation | Write one sentence about evaluation. | Evaluation measures how well a model performs... |  |
| 038 | decoder_generation | test | Write one sentence about tokenization. | Tokenization splits text into units such as words/subwords... |  |

## Suggested Preprocessing

- Filter by `task_type` before training so input and target fields are consistent.
- For classification, map labels such as `positive`, `neutral`, `negative` to integer ids.
- For generation, keep `target_text` as the supervised sequence.
- Do not train on validation or test rows.
- Add your own rows to make the task less toy-like.
